# 00 - Tokenizer analysis (Test 1 rerun via `lgmtp.boundaries`)
CPU only. Reproduces legacy Test 1 on FLORES+ `hin_Deva` devtest. Needs a Hugging Face login with access to `meta-llama/Llama-3.2-1B-Instruct` and `openlanguagedata/flores_plus`.

Regression targets: Llama-3.2-1B tokens/word 2.67, word-boundary cuts 38.6%, mid-akshara cuts 54.3%, k=4 inside-word 60.6% (each ±0.1).

**Qwen3.5-0.8B note.** The legacy run reported 3.33 tokens/word. That number came from an older transformers loading path that applied the Qwen2 pre-tokenizer regex (`\p{L}+`, which splits before every matra) instead of the model's own `tokenizer.json` regex (`[\p{L}\p{M}]+`). The real Qwen3.5 tokenizer gives about 2.16 tokens/word. Below we report it, and also reproduce the legacy 3.33 with the Qwen2 regex swapped in to confirm the explanation.

In [1]:
import json
from pathlib import Path
import pandas as pd
from lgmtp.boundaries import (HFTok, analyze, summarize, print_summary, render_example,
                              EXAMPLE_LEGEND, load_flores_hindi, legacy_qwen2_pretok_tokenizer)
RES = Path('../results/phase1'); RES.mkdir(parents=True, exist_ok=True)
KS = [2, 3, 4, 5, 6, 7, 8]; K_FOCUS = 4
LEGACY_QWEN = 'Qwen/Qwen3.5-0.8B [legacy Qwen2 regex]'
TOKS = [('meta-llama/Llama-3.2-1B-Instruct', HFTok('meta-llama/Llama-3.2-1B-Instruct')),
        ('Qwen/Qwen3.5-0.8B', HFTok('Qwen/Qwen3.5-0.8B')),
        (LEGACY_QWEN, HFTok(legacy_qwen2_pretok_tokenizer('Qwen/Qwen3.5-0.8B')))]

Linguistically-guided-MTP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
sents = load_flores_hindi('devtest')
print(len(sents), 'sentences; first:', sents[0])

1012 sentences; first: उन्होंने कहा “कि अब हमारे पास 4 महीने उम्र वाले चूहे हैं जिन्हें मधुमेह नहीं है जो मधुमेह के रोगी थे। ”


In [3]:
summaries, example_lines = [], []
for name, tk in TOKS:
    tbs = [tk.to_bytes(s) for s in sents]
    tot, pk = analyze(tbs, KS)
    S = summarize(name, tot, pk, KS)
    summaries.append(S); print_summary(S)
    example_lines += [f'### {name} (k={K_FOCUS})'] + [render_example(tb, K_FOCUS) for tb in tbs[:6]] + ['']
(RES / 'test1_summary.json').write_text(json.dumps(summaries, ensure_ascii=False, indent=2), encoding='utf-8')
(RES / 'test1_examples.txt').write_text(EXAMPLE_LEGEND + '\n' + '\n'.join(example_lines), encoding='utf-8')


=== meta-llama/Llama-3.2-1B-Instruct ===
sentences=1012  tokens=68511  tokens/word=2.67  tokens/akshara=0.80  tokens/char=0.52
tokens that START mid-akshara: 53.5%   mid-UTF-8: 0.1%
all token cuts -> word boundary 38.6% | mid-word 6.9% | mid-akshara 54.3% | mid-UTF-8 0.1%

          k       @word    mid-word     mid-aks    mid-utf8  inside-any words/draft     0-word%         fwd        back
          2        39.0         6.7        54.2         0.1        61.0        0.78        27.8        1.72        1.75
          3        39.3         6.6        54.1         0.1        60.7        1.17        11.9        1.71        1.75
          4        39.4         6.5        54.1         0.1        60.6        1.56         4.5        1.71        1.74
          5        39.4         6.5        54.1         0.1        60.6        1.96         1.5        1.71        1.73
          6        39.3         6.4        54.1         0.1        60.7        2.35         0.4        1.70        1.73
     


=== Qwen/Qwen3.5-0.8B ===
sentences=1012  tokens=55358  tokens/word=2.16  tokens/akshara=0.64  tokens/char=0.42
tokens that START mid-akshara: 18.9%   mid-UTF-8: 0.1%
all token cuts -> word boundary 51.2% | mid-word 29.5% | mid-akshara 19.2% | mid-UTF-8 0.1%

          k       @word    mid-word     mid-aks    mid-utf8  inside-any words/draft     0-word%         fwd        back
          2        52.1        28.9        19.0         0.1        47.9        1.02        26.5        1.87        1.92
          3        52.4        28.7        18.8         0.1        47.6        1.53        11.7        1.87        1.92
          4        52.7        28.5        18.7         0.1        47.3        2.04         4.4        1.87        1.91
          5        52.7        28.5        18.7         0.1        47.3        2.55         1.5        1.87        1.90
          6        52.8        28.5        18.6         0.1        47.2        3.06         0.5        1.87        1.90
          7        


=== Qwen/Qwen3.5-0.8B [legacy Qwen2 regex] ===
sentences=1012  tokens=85415  tokens/word=3.33  tokens/akshara=0.99  tokens/char=0.65
tokens that START mid-akshara: 45.8%   mid-UTF-8: 0.0%
all token cuts -> word boundary 33.0% | mid-word 20.6% | mid-akshara 46.4% | mid-UTF-8 0.0%

          k       @word    mid-word     mid-aks    mid-utf8  inside-any words/draft     0-word%         fwd        back
          2        33.3        20.3        46.3         0.0        66.7        0.65        39.6        2.20        2.24
          3        33.5        20.1        46.3         0.0        66.5        0.99        22.5        2.19        2.24
          4        33.6        20.1        46.2         0.0        66.4        1.31        11.5        2.18        2.24
          5        33.7        20.1        46.2         0.0        66.3        1.64         5.5        2.18        2.23
          6        33.7        20.0        46.2         0.0        66.3        1.97         2.5        2.18        2.2

10512

## Regression check against the legacy Test 1 numbers

In [4]:
EXPECTED = {
    'meta-llama/Llama-3.2-1B-Instruct': dict(tokens_per_word=2.67, pct_cuts_word_boundary=38.6,
                                             pct_cuts_mid_akshara=54.3, k4_inside_word=60.6),
    LEGACY_QWEN: dict(tokens_per_word=3.33),   # explains the legacy Qwen number
}
rows = []
for S in summaries:
    got = dict(tokens_per_word=S['tokens_per_word'], pct_cuts_word_boundary=S['pct_cuts_word_boundary'],
               pct_cuts_mid_akshara=S['pct_cuts_mid_akshara'],
               k4_inside_word=next(r for r in S['per_k'] if r['k'] == 4)['pct_end_anywhere_inside_word'])
    for k, exp in EXPECTED.get(S['tokenizer'], {}).items():
        nd = 2 if k == 'tokens_per_word' else 1          # legacy precision
        ok = abs(round(got[k], nd) - exp) <= (0.005 if nd == 2 else 0.1) + 1e-9
        rows.append(dict(tokenizer=S['tokenizer'], metric=k, expected=exp, got=round(got[k], 3), ok=ok))
reg = pd.DataFrame(rows); reg.to_csv(RES / 'test1_regression.csv', index=False); print(reg.to_string())
assert reg.ok.all(), 'Test 1 regression failed'
qwen = next(S for S in summaries if S['tokenizer'] == 'Qwen/Qwen3.5-0.8B')
print(f"Qwen3.5-0.8B (real tokenizer): {qwen['tokens_per_word']:.2f} tokens/word "
      "(the legacy 3.33 is reproduced only with the Qwen2 regex: a loading artifact)")

                                tokenizer                  metric  expected     got    ok
0        meta-llama/Llama-3.2-1B-Instruct         tokens_per_word      2.67   2.672  True
1        meta-llama/Llama-3.2-1B-Instruct  pct_cuts_word_boundary     38.60  38.629  True
2        meta-llama/Llama-3.2-1B-Instruct    pct_cuts_mid_akshara     54.30  54.297  True
3        meta-llama/Llama-3.2-1B-Instruct          k4_inside_word     60.60  60.641  True
4  Qwen/Qwen3.5-0.8B [legacy Qwen2 regex]         tokens_per_word      3.33   3.331  True
Qwen3.5-0.8B (real tokenizer): 2.16 tokens/word (the legacy 3.33 is reproduced only with the Qwen2 regex: a loading artifact)


In [5]:
print(EXAMPLE_LEGEND)
print('\n'.join(example_lines)[:3000])

Legend: | token boundary,  ‖ end of a fixed-k draft,  tag = what the draft ends on: W=word boundary, w=inside word, a=inside akshara, u=inside UTF-8 char, ␣ = space

### meta-llama/Llama-3.2-1B-Instruct (k=4)
उन | ्ह | ों | न ‖[a] े | ␣कह | ा | ␣“ ‖[W] क | ि | ␣अब | ␣हम ‖[a] ार | े | ␣प | ास ‖[W] ␣ | 4 | ␣मह | ीन ‖[a] े | ␣उम | ्र | ␣व ‖[a] ाल | े | ␣च | ूह ‖[a] े | ␣ह | ैं | ␣ज ‖[a] िन | ्ह | ें | ␣मध ‖[a] ुम | ेह | ␣नह | ीं ‖[W] ␣ह | ै | ␣ज | ो ‖[W] ␣मध | ुम | ेह | ␣क ‖[a] े | ␣र | ोग | ी ‖[W] ␣थ | े। | ␣”
ड | ॉ | . | ␣ए ‖[w] ह | ुड | ␣उ | र ‖[W] , | ␣न | ोव | ा ‖[W] ␣स | ्क | ोट | िय ‖[a] ा | ␣क | े | ␣ह ‖[a] ैल | िफ | ़ | ै ‖[w] क | ्स | ␣म | ें ‖[W] ␣ड | ल | ह | ौज ‖[a] ़ी | ␣व | िश | ्वव ‖[a] िद | ्य | ालय | ␣म ‖[a] ें | ␣च | िक | ित ‖[a] ्स | ा | ␣क | े ‖[W] ␣प | ्र | ोफ | ़े ‖[w] सर | ␣और | ␣कन | ाड ‖[a] ाई | ␣ड | ाय | ब ‖[a] िट | ीज | ़ | ␣एस ‖[a] ोस | िए | शन | ␣क ‖[a] े | ␣न | ैद | ान ‖[a] िक | ␣​​ | और | ␣व ‖[a] ैज | ्ञ | ान | िक ‖[W] ␣व | िभ | ाग | ␣क ‖[a] े | ␣अध | ्यक | 